In [15]:
import sqlite3
import csv
import datetime

# Creating SQLite Database

In [93]:
csv_file = "../2022_place_canvas_history.csv"
db_file = "2022_place_canvas.db"
table_name = "canvas_history"

conn = sqlite3.connect(db_file)
cursor = conn.cursor()

In [73]:
query = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        timestamp TEXT,
        user_id TEXT,
        pixel_color TEXT,
        coordinate TEXT
    )
"""
cursor.execute(query)

In [74]:
with open(csv_file, "rt") as f:
    reader = csv.reader(f)

    next(reader)

    query = f"""
        INSERT INTO {table_name}
        VALUES (?, ?, ?, ?)
    """

    cursor.executemany(query, reader)

In [27]:
query = f"""
    UPDATE {table_name}
    SET timestamp = DATETIME(SUBSTR(timestamp, 1, LENGTH(timestamp) - 4))
"""
cursor.execute(query)

In [94]:
query = f"""
    CREATE INDEX
        date_color
    ON
        {table_name}
    (timestamp, pixel_color)
"""
cursor.execute(query)

In [95]:
query = f"""
    CREATE INDEX
        date_coord
    ON
        {table_name}
    (timestamp, coordinate)
"""
cursor.execute(query)

In [96]:
query = f"""
    CREATE INDEX
        date_idx
    ON
        {table_name}
    (timestamp)
"""
cursor.execute(query)

In [113]:
conn.commit()
conn.close()

# Query Database

In [98]:
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

In [112]:
start_time = "2022-04-03 12:00:00"
end_time = "2022-04-03 13:00:00"

query = f"""
    SELECT
        pixel_color,
        COUNT(*) AS frequency
    FROM {table_name}
    WHERE timestamp >= DATETIME('{start_time}') AND timestamp <= DATETIME('{end_time}')
    GROUP BY pixel_color
    ORDER BY frequency DESC
    LIMIT 3
"""
conn.execute(query).fetchall()

[('#000000', 319297), ('#FFFFFF', 245000), ('#FF4500', 175582)]

In [111]:
query = f"""
    SELECT
        coordinate,
        COUNT(*) AS frequency
    FROM {table_name}
    WHERE timestamp >= ('{start_time}') AND timestamp <= ('{end_time}')
    GROUP BY coordinate
    ORDER BY frequency DESC
    LIMIT 3
"""
conn.execute(query).fetchall()

[('359,564', 914), ('349,564', 800), ('0,0', 687)]